# LeetCode 2103 — Rings and Rods bằng Python

**Link đề:** https://leetcode.com/problems/rings-and-rods/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

```python
class Solution:
    def countPoints(self, rings: str) -> int:
        ...
```

## Mục tiêu học
1. Đọc chuỗi theo **từng cặp ký tự** (màu, cọc): `range(0, n, 2)`, slicing bước nhảy `s[0::2]`, `zip`.
2. Đi từ cách theo gợi ý "**với mỗi cọc, quét cả chuỗi**" → **dict cọc → set màu** → **bitmask** (mỗi màu một bit) → mẹo kiểm tra chuỗi con.
3. Nắm cú pháp: `set`, `dict.setdefault`, `defaultdict(set)`, toán tử bit `|`, `<<`, `==`, `int(ch)`.
4. Trace từng cặp, chứng minh, Time/Space, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và cách hiểu đúng

Có `n` chiếc vòng, mỗi chiếc màu **R**ed, **G**reen hoặc **B**lue, được xỏ vào 10 cọc đánh số `0..9`.

Chuỗi `rings` dài `2n`; **mỗi 2 ký tự** mô tả một vòng: ký tự thứ nhất là **màu**, ký tự thứ hai là **số cọc**.

Ví dụ `"R3G2B1"` → vòng đỏ ở cọc 3, vòng xanh lá ở cọc 2, vòng xanh dương ở cọc 1.

Trả về **số cọc có đủ cả ba màu**.

| `rings` | Kết quả | Giải thích |
|---|:---:|---|
| `"B0B6G0R6R0R6G9"` | 1 | cọc 0 có B, G, R; cọc 6 chỉ có B, R; cọc 9 chỉ có G |
| `"B0R0G0R9R0B0G0"` | 1 | cọc 0 đủ ba màu (có lặp); cọc 9 chỉ R |
| `"G4"` | 0 | chỉ một vòng |

**Constraints:** `rings.length == 2n`, `1 <= n <= 100`; vị trí chẵn là màu `R/G/B`, vị trí lẻ là chữ số `0..9`.

Ý quan trọng: một cọc có thể có **nhiều vòng cùng màu** — ta chỉ quan tâm **có mặt** của từng màu, không phải số lượng.

In [1]:
examples = [("B0B6G0R6R0R6G9", 1), ("B0R0G0R9R0B0G0", 1), ("G4", 0)]
for s, exp in examples:
    print(f"{s!r:20} -> kỳ vọng {exp}")

'B0B6G0R6R0R6G9'     -> kỳ vọng 1
'B0R0G0R9R0B0G0'     -> kỳ vọng 1
'G4'                 -> kỳ vọng 0


## 2. Nền tảng Python cần dùng

### 2.1. Duyệt chuỗi theo bước 2: `range(0, len(s), 2)`

`i` nhận 0, 2, 4, ...; `s[i]` là màu, `s[i + 1]` là cọc.

In [2]:
s = "R3G2B1"
for i in range(0, len(s), 2):
    print(f"i={i}: màu={s[i]}, cọc={s[i + 1]}")

i=0: màu=R, cọc=3
i=2: màu=G, cọc=2
i=4: màu=B, cọc=1


### 2.2. Slicing có bước: `s[0::2]` và `s[1::2]`

`s[start::step]` lấy từ `start`, nhảy `step`. Ta tách được chuỗi màu và chuỗi cọc trong một dòng.

In [3]:
s = "B0B6G0R6R0R6G9"
print("màu :", s[0::2])
print("cọc :", s[1::2])

màu : BBGRRRG
cọc : 0606069


### 2.3. `zip(a, b)` — ghép từng cặp phần tử

`zip("BBG", "060")` cho `('B','0'), ('B','6'), ('G','0')`. Kết hợp với slicing ở trên → duyệt từng vòng rất gọn.

In [4]:
s = "B0B6G0"
for color, rod in zip(s[0::2], s[1::2]):
    print(color, rod)

B 0
B 6
G 0


### 2.4. `int(ch)` — đổi ký tự chữ số thành số

`int("6") == 6`. Dùng để làm chỉ số list `rods[6]`. (Cách khác: `ord(ch) - ord("0")`.)

In [5]:
print(int("6"), ord("6") - ord("0"), type(int("6")))

6 6 <class 'int'>


### 2.5. `set` — tập các màu đã có, tự bỏ trùng

In [6]:
colors = set()
for c in "BGRRB":
    colors.add(c)
print(colors, len(colors), len(colors) == 3)
print({"R", "G", "B"} == {"B", "R", "G"})   # set không quan tâm thứ tự

{'B', 'G', 'R'} 3 True
True


### 2.6. `dict.setdefault` và `defaultdict(set)`

Muốn `rod -> set màu`, mỗi lần gặp cọc mới phải tạo set rỗng. Hai cách gọn:

- `d.setdefault(key, set()).add(x)`: nếu key chưa có thì gán `set()` rồi trả về nó.
- `defaultdict(set)`: tự tạo `set()` khi truy cập key mới.

In [7]:
from collections import defaultdict
d = {}
d.setdefault("0", set()).add("B")
d.setdefault("0", set()).add("G")
print(d)

dd = defaultdict(set)
dd["6"].add("R"); dd["6"].add("R")
print(dict(dd))

{'0': {'B', 'G'}}
{'6': {'R'}}


### 2.7. Toán tử bit: `1 << k`, `|`, so sánh với `7`

Gán mỗi màu một bit: `R = 1 (001)`, `G = 2 (010)`, `B = 4 (100)`. Phép **OR** `|` bật bit. Đủ ba màu ⇔ mask `== 7 (111)`.

In [8]:
BIT = {"R": 1, "G": 2, "B": 4}
mask = 0
for c in "BRB":
    mask |= BIT[c]
    print(f"thêm {c}: mask = {mask} = {mask:03b}")
print("đủ 3 màu?", mask == 7)
mask |= BIT["G"]
print(f"thêm G: mask = {mask} = {mask:03b} -> đủ 3 màu? {mask == 7}")

thêm B: mask = 4 = 100
thêm R: mask = 5 = 101
thêm B: mask = 5 = 101
đủ 3 màu? False
thêm G: mask = 7 = 111 -> đủ 3 màu? True


## 3. Cách 1 — Trực giác (theo gợi ý): với mỗi cọc, quét cả chuỗi

Gợi ý của đề: *"For every rod, look through rings… Create 3 booleans, 1 for each color."*

Với `rod` từ `0` đến `9`: đặt `has_r = has_g = has_b = False`, quét mọi cặp; nếu cặp thuộc cọc này thì bật boolean của màu đó. Cuối cùng nếu cả ba đều `True` thì cộng 1.

In [9]:
def count_points_per_rod(rings: str) -> int:
    total = 0
    for rod in "0123456789":
        has_r = has_g = has_b = False
        for i in range(0, len(rings), 2):
            if rings[i + 1] == rod:
                if rings[i] == "R":
                    has_r = True
                elif rings[i] == "G":
                    has_g = True
                else:
                    has_b = True
        if has_r and has_g and has_b:
            total += 1
    return total

for s, exp in examples:
    print(count_points_per_rod(s), "kỳ vọng", exp)

1 kỳ vọng 1
1 kỳ vọng 1
0 kỳ vọng 0


## 4. Nút thắt của cách quét theo từng cọc

Chuỗi được quét **10 lần** (mỗi cọc một lần) → `O(10·n)`. Vì 10 là hằng số nên vẫn là `O(n)`, nhưng:

- Nếu số cọc là `m` (không cố định 10) thì thành `O(m·n)`.
- Mỗi lượt quét, phần lớn cặp **không thuộc cọc đang xét** → công sức bị lãng phí.

**Cải tiến:** chỉ quét chuỗi **một lần**; mỗi cặp `(màu, cọc)` cập nhật ngay "thông tin của cọc đó". Cuối cùng đếm các cọc đủ màu.

## 5. Bước đột phá — Mỗi cọc giữ một "tập màu" và cập nhật khi gặp vòng

Ta cần cho mỗi cọc biết **những màu nào đã có**. Có hai cách lưu:

1. **Set màu**: `rods[rod].add(color)`; đủ khi `len(...) == 3`.
2. **Bitmask** 3 bit: `mask[rod] |= BIT[color]`; đủ khi `mask[rod] == 7`. Gọn và nhanh vì chỉ là số nguyên.

Ví dụ `"B0B6G0R6R0R6G9"`:

```text
B0 -> cọc 0: {B}
B6 -> cọc 6: {B}
G0 -> cọc 0: {B,G}
R6 -> cọc 6: {B,R}
R0 -> cọc 0: {B,G,R}  ✅
R6 -> cọc 6: {B,R} (R đã có)
G9 -> cọc 9: {G}
```

## 6. Lời giải chính (bitmask) — mổ xẻ rồi ghép

### 6.1. Đoạn 1: bảng bit cho màu, mảng 10 mask cho 10 cọc

In [10]:
BIT = {"R": 1, "G": 2, "B": 4}
masks = [0] * 10
print(BIT, masks)

{'R': 1, 'G': 2, 'B': 4} [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### 6.2. Đoạn 2: cập nhật cho từng vòng

In [11]:
for color, rod in zip("B0G0R0"[0::2], "B0G0R0"[1::2]):
    masks[int(rod)] |= BIT[color]
print(masks)   # cọc 0 = 7

[7, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### 6.3. Đoạn 3: đếm cọc có mask == 7, rồi ghép thành hàm

In [12]:
def count_points(rings: str) -> int:
    BIT = {"R": 1, "G": 2, "B": 4}
    masks = [0] * 10
    for i in range(0, len(rings), 2):
        masks[int(rings[i + 1])] |= BIT[rings[i]]
    return sum(1 for m in masks if m == 7)

for s, exp in examples:
    print(count_points(s), "kỳ vọng", exp)

1 kỳ vọng 1
1 kỳ vọng 1
0 kỳ vọng 0


### 6.4. Tại sao dùng `|=` mà không dùng `+=`?

`|=` là **bật bit**: bật lại bit đã bật không đổi gì. `+=` thì cộng dồn: hai vòng đỏ cho `1 + 1 = 2` — trông giống "có màu G" → sai. Xem Lỗi 2 ở mục 12.

In [13]:
print("1 | 1 =", 1 | 1, "| 1 + 1 =", 1 + 1)

1 | 1 = 1 | 1 + 1 = 2


## 7. TRACE từng bước

### 7.1. Bảng trace — `rings = "B0B6G0R6R0R6G9"`

| Cặp | Màu | Cọc | Bit | mask[cọc] trước | mask[cọc] sau |
|:---:|:---:|:---:|:---:|:---:|:---:|
| B0 | B | 0 | 100 | 000 | 100 |
| B6 | B | 6 | 100 | 000 | 100 |
| G0 | G | 0 | 010 | 100 | 110 |
| R6 | R | 6 | 001 | 100 | 101 |
| R0 | R | 0 | 001 | 110 | **111** |
| R6 | R | 6 | 001 | 101 | 101 (không đổi) |
| G9 | G | 9 | 010 | 000 | 010 |

Cuối: chỉ cọc 0 có `111 = 7` → đáp án 1.

In [14]:
def trace_count_points(rings):
    BIT = {"R": 1, "G": 2, "B": 4}
    masks = [0] * 10
    for i in range(0, len(rings), 2):
        color, rod = rings[i], int(rings[i + 1])
        before = masks[rod]
        masks[rod] |= BIT[color]
        note = " (không đổi: màu đã có)" if before == masks[rod] else ""
        print(f"  cặp {rings[i:i+2]}: cọc {rod}: {before:03b} -> {masks[rod]:03b}{note}")
    full = [r for r in range(10) if masks[r] == 7]
    print("  cọc đủ màu:", full, "-> đáp án", len(full))
    return len(full)

trace_count_points("B0B6G0R6R0R6G9")

  cặp B0: cọc 0: 000 -> 100
  cặp B6: cọc 6: 000 -> 100
  cặp G0: cọc 0: 100 -> 110
  cặp R6: cọc 6: 100 -> 101
  cặp R0: cọc 0: 110 -> 111
  cặp R6: cọc 6: 101 -> 101 (không đổi: màu đã có)
  cặp G9: cọc 9: 000 -> 010
  cọc đủ màu: [0] -> đáp án 1


1

### 7.2. Nhánh đặc biệt: nhiều vòng cùng màu trên một cọc

In [15]:
trace_count_points("B0R0G0R9R0B0G0")

  cặp B0: cọc 0: 000 -> 100
  cặp R0: cọc 0: 100 -> 101
  cặp G0: cọc 0: 101 -> 111
  cặp R9: cọc 9: 000 -> 001
  cặp R0: cọc 0: 111 -> 111 (không đổi: màu đã có)
  cặp B0: cọc 0: 111 -> 111 (không đổi: màu đã có)
  cặp G0: cọc 0: 111 -> 111 (không đổi: màu đã có)
  cọc đủ màu: [0] -> đáp án 1


1

### 7.3. Trace cách dict → set

In [16]:
def trace_dict_set(rings):
    rods = {}
    for color, rod in zip(rings[0::2], rings[1::2]):
        rods.setdefault(rod, set()).add(color)
        print(f"  {color}{rod}: rods = { {k: ''.join(sorted(v)) for k, v in rods.items()} }")
    return sum(1 for v in rods.values() if len(v) == 3)

print("đáp án:", trace_dict_set("B0B6G0R6R0R6G9"))

  B0: rods = {'0': 'B'}
  B6: rods = {'0': 'B', '6': 'B'}
  G0: rods = {'0': 'BG', '6': 'B'}
  R6: rods = {'0': 'BG', '6': 'BR'}
  R0: rods = {'0': 'BGR', '6': 'BR'}
  R6: rods = {'0': 'BGR', '6': 'BR'}
  G9: rods = {'0': 'BGR', '6': 'BR', '9': 'G'}
đáp án: 1


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Mỗi cọc quét cả chuỗi với 3 boolean (theo gợi ý)

In [17]:
class Solution:
    def countPoints(self, rings: str) -> int:
        total = 0
        for rod in "0123456789":
            has = {"R": False, "G": False, "B": False}
            for i in range(0, len(rings), 2):
                if rings[i + 1] == rod:
                    has[rings[i]] = True
            if all(has.values()):
                total += 1
        return total

print([Solution().countPoints(s) for s, _ in examples])

[1, 1, 0]


`all(iterable)` trả `True` nếu **mọi** phần tử đều đúng — gọn hơn viết `a and b and c`.

### Cách B — dict cọc → set màu

In [18]:
from collections import defaultdict

class Solution:
    def countPoints(self, rings: str) -> int:
        rods = defaultdict(set)
        for color, rod in zip(rings[0::2], rings[1::2]):
            rods[rod].add(color)
        return sum(1 for colors in rods.values() if len(colors) == 3)

print([Solution().countPoints(s) for s, _ in examples])

[1, 1, 0]


### Cách C — Bitmask (lời giải chính)

In [19]:
class Solution:
    def countPoints(self, rings: str) -> int:
        bit = {"R": 1, "G": 2, "B": 4}
        masks = [0] * 10
        for i in range(0, len(rings), 2):
            masks[int(rings[i + 1])] |= bit[rings[i]]
        return masks.count(7)

print([Solution().countPoints(s) for s, _ in examples])

[1, 1, 0]


### Cách D — Ba tập cọc theo màu rồi lấy giao

`red = {cọc có vòng đỏ}`, tương tự `green`, `blue`. Cọc đủ màu ⇔ thuộc cả ba tập ⇔ thuộc `red & green & blue`.

In [20]:
class Solution:
    def countPoints(self, rings: str) -> int:
        by_color = {"R": set(), "G": set(), "B": set()}
        for color, rod in zip(rings[0::2], rings[1::2]):
            by_color[color].add(rod)
        return len(by_color["R"] & by_color["G"] & by_color["B"])

print([Solution().countPoints(s) for s, _ in examples])

[1, 1, 0]


### Cách E — Mẹo chuỗi con: cọc `d` đủ màu ⇔ `"R"+d`, `"G"+d`, `"B"+d` đều nằm trong chuỗi

Vì sao không bị khớp "lệch cặp"? Chuỗi con `"Rd"` là **màu rồi chữ số**. Màu chỉ đứng ở vị trí **chẵn**, nên `"Rd"` chỉ có thể xuất hiện ở vị trí chẵn — tức đúng là một cặp thật. (Một chuỗi con lệch sẽ có dạng chữ số rồi màu.)

In [21]:
class Solution:
    def countPoints(self, rings: str) -> int:
        return sum(all(c + d in rings for c in "RGB") for d in "0123456789")

print([Solution().countPoints(s) for s, _ in examples])
print("Chuỗi lệch '0R' có trong 'B0R6'?", "0R" in "B0R6", "-> nhưng ta không bao giờ tìm dạng chữ số+màu")

[1, 1, 0]
Chuỗi lệch '0R' có trong 'B0R6'? True -> nhưng ta không bao giờ tìm dạng chữ số+màu


## 9. Vì sao thuật toán đúng?

**Invariant (Cách C):** sau khi xử lý `j` cặp đầu, bit màu `c` của `masks[r]` bật **khi và chỉ khi** trong `j` cặp đó có ít nhất một vòng màu `c` ở cọc `r`.

- *Khởi tạo:* mọi mask = 0, chưa có vòng nào — đúng.
- *Duy trì:* cặp `(c, r)` chỉ bật bit của `c` trong `masks[r]`; `|` không tắt bit nào, không đụng cọc khác.
- *Kết thúc:* `masks[r] == 7` ⇔ cả ba bit bật ⇔ cọc `r` có đủ ba màu.

Mỗi cọc được đếm đúng một lần (duyệt 10 phần tử của `masks`), nên không đếm trùng.

Cách B và D là cùng ý tưởng, chỉ thay "bit" bằng "phần tử của set". Cách E đúng nhờ lập luận vị trí chẵn ở trên.

## 10. Phân tích Time / Space Complexity

Gọi `n` là số vòng (`len(rings) = 2n`), `m = 10` số cọc, `C = 3` số màu.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. Quét theo cọc | `O(m·n)` = `O(10n)` | `O(1)` | theo gợi ý đề |
| B. dict → set | `O(n)` | `O(m·C)` | dễ đọc |
| C. Bitmask | `O(n)` | `O(m)` (10 số nguyên) | gọn, nhanh |
| D. Ba set giao | `O(n)` | `O(m·C)` | tư duy tập hợp |
| E. Chuỗi con | `O(m·C·n)` | `O(1)` | mẹo hay, mỗi `in` quét chuỗi |

Với `m = 10`, `C = 3` cố định, mọi cách đều `O(n)`; khác nhau ở hằng số và độ tổng quát.

## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [22]:
import random
from collections import defaultdict

def sol_A(s): return count_points_per_rod(s)
def sol_B(s):
    rods = defaultdict(set)
    for c, r in zip(s[0::2], s[1::2]): rods[r].add(c)
    return sum(len(v) == 3 for v in rods.values())
def sol_C(s): return count_points(s)
def sol_D(s):
    by = {"R": set(), "G": set(), "B": set()}
    for c, r in zip(s[0::2], s[1::2]): by[c].add(r)
    return len(by["R"] & by["G"] & by["B"])
def sol_E(s): return sum(all(c + d in s for c in "RGB") for d in "0123456789")

SOLS = [sol_A, sol_B, sol_C, sol_D, sol_E]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:26} PASS ({actual})")

tests = examples + [
    ("R0G0B0", 1),
    ("R0G1B2", 0),                      # ba màu nhưng ở ba cọc khác nhau
    ("R0R0R0", 0),                      # nhiều vòng nhưng chỉ một màu
    ("".join(c + d for d in "0123456789" for c in "RGB"), 10),   # mọi cọc đủ màu
    ("R9G9B9R0G0", 1),
]
for i, (s, exp) in enumerate(tests):
    for f in SOLS:
        check(f"test {i} {f.__name__}", f(s), exp)

test 0 sol_A               PASS (1)
test 0 sol_B               PASS (1)
test 0 sol_C               PASS (1)
test 0 sol_D               PASS (1)
test 0 sol_E               PASS (1)
test 1 sol_A               PASS (1)
test 1 sol_B               PASS (1)
test 1 sol_C               PASS (1)
test 1 sol_D               PASS (1)
test 1 sol_E               PASS (1)
test 2 sol_A               PASS (0)
test 2 sol_B               PASS (0)
test 2 sol_C               PASS (0)
test 2 sol_D               PASS (0)
test 2 sol_E               PASS (0)
test 3 sol_A               PASS (1)
test 3 sol_B               PASS (1)
test 3 sol_C               PASS (1)
test 3 sol_D               PASS (1)
test 3 sol_E               PASS (1)
test 4 sol_A               PASS (0)
test 4 sol_B               PASS (0)
test 4 sol_C               PASS (0)
test 4 sol_D               PASS (0)
test 4 sol_E               PASS (0)
test 5 sol_A               PASS (0)
test 5 sol_B               PASS (0)
test 5 sol_C               P

In [23]:
rng = random.Random(2103)
for _ in range(500):
    s = "".join(rng.choice("RGB") + rng.choice("0123456789") for _ in range(rng.randint(1, 30)))
    outs = [f(s) for f in SOLS]
    assert len(set(outs)) == 1, (s, outs)
print("500 test ngẫu nhiên: 5 cách khớp nhau")

500 test ngẫu nhiên: 5 cách khớp nhau


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Đếm số vòng thay vì số màu khác nhau

In [24]:
def wrong_count_rings(rings):
    cnt = [0] * 10
    for i in range(0, len(rings), 2):
        cnt[int(rings[i + 1])] += 1
    return sum(1 for c in cnt if c >= 3)       # SAI: 3 vòng chưa chắc 3 màu

print("Sai :", wrong_count_rings("B0B6G0R6R0R6G9"), "| Đúng:", count_points("B0B6G0R6R0R6G9"))

Sai : 2 | Đúng: 1


**Vì sao sai:** cọc 6 có 3 vòng (B, R, R) nhưng chỉ 2 màu. **Sửa:** lưu *tập màu* (set/bitmask).

### Lỗi 2: Dùng `+=` thay vì `|=` với bitmask

In [25]:
def wrong_plus(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    masks = [0] * 10
    for i in range(0, len(rings), 2):
        masks[int(rings[i + 1])] += bit[rings[i]]   # SAI
    return masks.count(7)

print("Sai :", wrong_plus("R0R0R0B0"), "| Đúng:", count_points("R0R0R0B0"))   # 1+1+1+4 = 7 !

Sai : 1 | Đúng: 0


### Lỗi 3: Duyệt từng ký tự (bước 1) thay vì từng cặp

In [26]:
def wrong_step1(rings):
    rods = defaultdict(set)
    for i in range(len(rings) - 1):               # SAI: bước 1 -> có cặp lệch "0R"
        rods[rings[i + 1]].add(rings[i])
    return sum(len(v) == 3 for v in rods.values())

print("Sai :", wrong_step1("R0G1R2G3R4G5"), "| Đúng:", count_points("R0G1R2G3R4G5"))
print("Ví dụ key lệch:", "cặp lệch ('0','G'), ('2','G'), ('4','G') làm 'G' trông như một cọc có 3 'màu'")

Sai : 1 | Đúng: 0
Ví dụ key lệch: cặp lệch ('0','G'), ('2','G'), ('4','G') làm 'G' trông như một cọc có 3 'màu'


**Vì sao sai:** với bước 1, các "cặp lệch" như `("0", "G")` cũng được thêm, làm hỏng dữ liệu (thêm cả key là chữ cái, value là chữ số). **Sửa:** `range(0, len(rings), 2)`.

### Lỗi 4: Dùng ký tự `'0'` làm chỉ số list

In [27]:
masks = [0] * 10
try:
    masks["0"] |= 1
except TypeError as e:
    print("TypeError:", e, "-> sửa: masks[int('0')]")

TypeError: list indices must be integers or slices, not str -> sửa: masks[int('0')]


## 13. Mẹo nhận dạng pattern

- Chuỗi mã hóa theo **cụm cố định** (mỗi 2/3 ký tự một bản ghi) → `range(0, n, 2)` hoặc `zip(s[0::2], s[1::2])`.
- Câu hỏi "**có đủ** tất cả loại không" (không quan tâm số lượng) → **set** hoặc **bitmask**; đủ khi `len == số loại` hoặc `mask == (1 << số_loại) - 1`.
- Số nhóm nhỏ, cố định (10 cọc, 26 chữ cái) → mảng thay cho dict.

Bài liên quan: 1832 Check if the Sentence Is Pangram (bitmask 26 bit), 771 Jewels and Stones, 2287 Rearrange Characters to Make Target String.

## 14. Biến thể

### 14.1. Trả về danh sách cọc đủ màu (tăng dần)

In [28]:
def full_rods(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    masks = [0] * 10
    for c, r in zip(rings[0::2], rings[1::2]):
        masks[int(r)] |= bit[c]
    return [r for r in range(10) if masks[r] == 7]

print(full_rods("B0B6G0R6R0R6G9G6"))

[0, 6]


### 14.2. Tổng quát: `k` màu bất kỳ (chữ cái in hoa), cọc đủ **mọi** màu đã xuất hiện trong chuỗi

In [29]:
def count_full_general(rings):
    colors = sorted(set(rings[0::2]))
    bit = {c: 1 << i for i, c in enumerate(colors)}
    full = (1 << len(colors)) - 1
    masks = defaultdict(int)
    for c, r in zip(rings[0::2], rings[1::2]):
        masks[r] |= bit[c]
    return sum(m == full for m in masks.values())

print(count_full_general("A1B1C1D1A2B2"), count_full_general("X0Y0"))

1 1


### 14.3. Số vòng tối thiểu cần thêm để mọi cọc **đã có vòng** đều đủ 3 màu

In [30]:
def rings_to_add(rings):
    rods = defaultdict(set)
    for c, r in zip(rings[0::2], rings[1::2]):
        rods[r].add(c)
    return sum(3 - len(v) for v in rods.values())

print(rings_to_add("B0B6G0R6R0R6G9"))   # cọc 0:0, cọc 6:1, cọc 9:2 -> 3

3


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Tách thành danh sách cặp
`pairs("R3G2B1") -> [("R","3"), ("G","2"), ("B","1")]`

In [31]:
def pairs(rings):
    # TODO: viết code ở đây
    pass

# assert pairs("R3G2B1") == [("R", "3"), ("G", "2"), ("B", "1")]

### Bài 2 — Số màu khác nhau trên cọc `rod`

In [32]:
def colors_on_rod(rings, rod):
    # TODO: viết code ở đây
    pass

# assert colors_on_rod("B0B6G0R6R0R6G9", 6) == 2

### Bài 3 — Tự viết lại lời giải #2103 bằng bitmask

In [33]:
def count_points_ex(rings):
    # TODO: viết code ở đây
    pass

# assert count_points_ex("B0R0G0R9R0B0G0") == 1

### Bài 4 — Biến thể: cọc có **đúng hai** màu

In [34]:
def rods_exactly_two(rings):
    # TODO: viết code ở đây
    pass

# assert rods_exactly_two("B0B6G0R6R0R6G9") == 1   # cọc 6

### Bài 5 — Tự tạo test làm lộ **Lỗi 2** (`+=`)

In [35]:
my_tests = [
    # TODO: các chuỗi rings mà wrong_plus(s) != count_points(s)
]
# for s in my_tests:
#     assert wrong_plus(s) != count_points(s)

### Bài 6 — Mini challenge: Pangram bằng bitmask
`is_pangram(sentence)` trả True nếu câu (chữ thường) chứa đủ 26 chữ cái. Dùng một số nguyên làm mask 26 bit.

In [36]:
def is_pangram(sentence):
    # TODO: viết code ở đây
    pass

# assert is_pangram("thequickbrownfoxjumpsoverthelazydog") and not is_pangram("leetcode")

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | slicing bước 2 | `zip` hai slice | `list(zip(s[0::2], s[1::2]))` |
| 2 | lọc theo cọc | set màu | `len({c for c, r in pairs(s) if r == str(rod)})` |
| 3 | 10 mask | `|=` bit màu | `masks.count(7)` |
| 4 | đếm bit bật | `bin(m).count("1") == 2` | hoặc dùng set `len == 2` |
| 5 | `+=` cộng dồn khi lặp màu | cần màu lặp | `"R0R0R0B0"` cho tổng 7 |
| 6 | 26 bit | `1 << (ord(ch) - ord('a'))` | `mask == (1 << 26) - 1` |

## 17. Đáp án tham khảo

In [37]:
def pairs(rings):
    return list(zip(rings[0::2], rings[1::2]))

def colors_on_rod(rings, rod):
    return len({c for c, r in pairs(rings) if r == str(rod)})

def count_points_ex(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    masks = [0] * 10
    for c, r in pairs(rings):
        masks[int(r)] |= bit[c]
    return masks.count(7)

def rods_exactly_two(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    masks = [0] * 10
    for c, r in pairs(rings):
        masks[int(r)] |= bit[c]
    return sum(1 for m in masks if bin(m).count("1") == 2)

my_tests = ["R0R0R0B0", "G0G0G0R0", "R0R0B0B0G0"]

def is_pangram(sentence):
    mask = 0
    for ch in sentence:
        mask |= 1 << (ord(ch) - ord("a"))
    return mask == (1 << 26) - 1

assert pairs("R3G2B1") == [("R", "3"), ("G", "2"), ("B", "1")]
assert colors_on_rod("B0B6G0R6R0R6G9", 6) == 2
assert count_points_ex("B0R0G0R9R0B0G0") == 1
assert rods_exactly_two("B0B6G0R6R0R6G9") == 1
for s in my_tests:
    assert wrong_plus(s) != count_points(s), s
assert is_pangram("thequickbrownfoxjumpsoverthelazydog") and not is_pangram("leetcode")
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1–2: tách cặp bằng slicing + zip; số màu khác nhau = kích thước set.
- Bài 3: đúng lời giải chính.
- Bài 4: số bit bật trong mask = số màu; `bin(m)` cho chuỗi nhị phân, đếm ký tự `"1"`.
- Bài 5: `+=` sai khi một màu lặp lại tạo ra tổng trùng với mask khác (ví dụ 1+1+1+4 = 7, 2+2+2+1 = 7), hoặc khi đủ màu nhưng có lặp thì tổng vượt 7 (1+1+4+4+2 = 12).
- Bài 6: cùng kỹ thuật bitmask với 26 loại thay vì 3.

## 18. Tổng kết — template + checklist

**Template "nhóm theo khóa, kiểm tra đủ loại":**
```python
masks = [0] * số_nhóm
for loại, nhóm in zip(s[0::2], s[1::2]):
    masks[int(nhóm)] |= 1 << chỉ_số(loại)
return masks.count((1 << số_loại) - 1)
```

**Checklist trước khi submit:**
- [ ] Duyệt theo **cặp** (bước 2), không theo từng ký tự.
- [ ] Đếm **màu khác nhau**, không đếm số vòng.
- [ ] Bitmask dùng `|=`, không `+=`.
- [ ] Đổi ký tự cọc sang `int` trước khi làm chỉ số list.
- [ ] Đúng tên method `countPoints`.
- [ ] Ví dụ `"G4"` trả 0.